# Specular-Gaussians — Synthetic Specular Dataset Pipeline

> **Kernel**: `thesis_env` (Set via Kernel → Change Kernel after running `bosch_setup_thesis.ipynb` once to register it)

Runs the full Specular-Gaussians training sweep on the 8 Synthetic Specular scenes inside the BOSCH server environment, reading datasets directly from `/home/ghp4hc/datasets/datasets/synthetic_specular`.

**What this notebook does:**
1. **Proxy & Env Check**: Sets up environment variables for the BOSCH server.
2. **Imports & Self-Healing Deps**: Verifies `torch`, `lpips`, `torch_scatter`, and the anchor-based `diff_gaussian_rasterization` fork (with `visible_filter`), rebuilding/reinstalling any that are missing or mismatched.
3. **Dataset Verification**: Verifies the presence of the Synthetic Specular dataset root `/home/ghp4hc/datasets/datasets/synthetic_specular`.
4. **Dataset Validation**: Verifies `transforms_train.json` or `transforms.json` exists for all 8 scenes (`ashtray`, `dishes`, `headphone`, `jupyter`, `lock`, `plane`, `record`, `teapot`).
5. **Run Sweep**: Invokes `run_synthetic_specular.sh` in `Specular-Gaussians` with `DATA_ROOT=/home/ghp4hc/datasets/datasets/synthetic_specular`, logging outputs to `synthetic_specular_specular_gaussians_run.log`.
6. **Results Aggregation**: Formats and prints quantitative metrics (`results.json`) and final Gaussian point counts in a neat table.
7. **Output Archiving**: Zips the results to `specular_gaussians_output_synthetic_specular.zip` in the parent directory.
8. **Hugging Face Upload & Auto-Cleanup**: Uploads the zipped output results to `DiBiay/specular_gaussians-synthetic_specular-result`, then purges local zip & output folder to save disk space.

## c00 — Proxy Settings
Sets the BOSCH proxy for external connectivity.

In [ ]:
# ── Proxy (required for HF / huggingface cache / diagnostic endpoints) ────────
import os

PROXY = 'http://rb-proxy-sl.bosch.com:8080'
HOME  = os.path.expanduser('~')

os.environ['http_proxy']  = PROXY
os.environ['https_proxy'] = PROXY
os.environ['HTTP_PROXY']  = PROXY
os.environ['HTTPS_PROXY'] = PROXY

print(f'Proxy set to: {PROXY}')

## c01 — Config & Kernel Check
Defines paths and double-checks if the correct virtual environment kernel is loaded.

In [ ]:
# ── Configurations & environment variables check ─────────────────────────────
import os
import sys

HOME = os.path.expanduser('~')

# Robustly resolve Specular-Gaussians repository root path
if os.path.isdir('/home/ghp4hc/thesis-all/Specular-Gaussians'):
    REPO_ROOT = '/home/ghp4hc/thesis-all/Specular-Gaussians'
elif os.path.isdir(os.path.join(os.getcwd(), 'thesis-all', 'Specular-Gaussians')):
    REPO_ROOT = os.path.join(os.getcwd(), 'thesis-all', 'Specular-Gaussians')
else:
    REPO_ROOT = os.path.join(os.getcwd(), 'Specular-Gaussians')

ENV_NAME = 'thesis_env'

print(f'Active Python      : {sys.executable}')
print(f'Active Kernel name : {ENV_NAME}')
print(f'Repository Root    : {REPO_ROOT}')

assert REPO_ROOT in sys.executable or ENV_NAME in sys.executable or '.conda' in sys.executable, \
    f"WARNING: You are not running on the '{ENV_NAME}' kernel! Please select Kernel -> Change Kernel -> Python ({ENV_NAME})"

## c02 — Imports & GPU Validation
Verifies hardware detection and self-heals the compiled custom modules (`lpips`, `torch_scatter`, the anchor-based `diff_gaussian_rasterization` fork).

In [ ]:
# ── Verification of PyTorch & custom submodules ────────────────────────────────
import subprocess
import sys
import os
import glob
import torch

def ensure_importable(pkg, import_name=None, extra_pip_args=None):
    """Import a package, reinstalling it if the import fails (missing or broken binary)."""
    import_name = import_name or pkg
    try:
        __import__(import_name)
        return True
    except Exception as e:
        print(f'{import_name}: reinstalling ({e.__class__.__name__}: {e})')
        subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', pkg], capture_output=True)
        r = subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '--proxy', os.environ.get('HTTPS_PROXY', '')]
                           + (extra_pip_args or []), capture_output=True, text=True)
        print(r.stdout[-500:] if r.returncode == 0 else r.stderr[-500:])
        return r.returncode == 0

# Same nvcc search as bosch_setup_thesis.ipynb's c03_cuda_search (proven to work on this server):
# tries module-load first, then falls back to scanning common HPC mount points.
CUDA_SEARCH_SCRIPT = r'''
which nvcc 2>/dev/null && exit 0
for init in /etc/profile /etc/profile.d/modules.sh \
            /usr/share/lmod/lmod/init/bash /usr/share/lmod/lmod/init/sh; do
    [ -f "$init" ] && source "$init" 2>/dev/null
done
for mod in cuda/11.7 cuda/11.8 cuda/11.2 cuda/11 cuda CUDA/11.7 CUDA/11.8 CUDA cuda-11.7 cuda-11.8 cuda-11 cuda/12.6 cuda/12.1 cuda/12 cuda CUDA/12.6 CUDA/12.1 cuda-12.6 cuda-12-1 cuda-12; do
    module load "$mod" 2>/dev/null
    nv=$(which nvcc 2>/dev/null); [ -n "$nv" ] && echo "$nv" && exit 0
done
for base in /fs /work /gpfs /scratch /software /apps /appl /tools /opt/software /usr/local; do
    [ -d "$base" ] || continue
    result=$(find "$base" -name nvcc -type f -maxdepth 8 2>/dev/null | head -1)
    [ -n "$result" ] && echo "$result" && exit 0
done
'''

def find_cuda_home():
    cuda_home = os.environ.get('CUDA_HOME', '')
    if cuda_home and os.path.isfile(os.path.join(cuda_home, 'bin', 'nvcc')):
        return cuda_home
    r_nvcc = subprocess.run(['bash', '-c', CUDA_SEARCH_SCRIPT], capture_output=True, text=True, timeout=90)
    nvcc_path = r_nvcc.stdout.strip()
    if not nvcc_path or not os.path.isfile(nvcc_path):
        raise RuntimeError('nvcc not found — check CUDA module availability on server')
    return os.path.dirname(os.path.dirname(nvcc_path))

def build_and_install_rasterizer():
    """Build the anchor-based diff_gaussian_rasterization fork (with visible_filter) from source."""
    cuda_home = find_cuda_home()
    cc = f'{torch.cuda.get_device_capability(0)[0]}.{torch.cuda.get_device_capability(0)[1]}' if torch.cuda.is_available() else '8.0'
    src_dir = os.path.join(REPO_ROOT, 'submodules', 'depth-diff-gaussian-rasterization')
    print(f'Building diff_gaussian_rasterization from {src_dir} (CUDA_HOME={cuda_home}, arch={cc}) ...')
    build_cmd = (
        f'cd "{src_dir}" && rm -rf build dist *.egg-info && '
        f'CUDA_HOME={cuda_home} PATH={cuda_home}/bin:$PATH TORCH_CUDA_ARCH_LIST={cc} '
        f'{sys.executable} setup.py bdist_wheel 2>&1'
    )
    r = subprocess.run(build_cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print('BUILD FAILED:', r.stdout[-1500:])
        return False
    wheels = glob.glob(f'{src_dir}/dist/*.whl')
    if not wheels:
        print('No wheel produced')
        return False
    ri = subprocess.run([sys.executable, '-m', 'pip', 'install', wheels[0], '--no-deps', '--force-reinstall'],
                        capture_output=True, text=True)
    print(ri.stdout[-500:] if ri.returncode == 0 else ri.stderr[-500:])
    return ri.returncode == 0

print('PyTorch version :', torch.__version__)
print('CUDA Available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU device name :', torch.cuda.get_device_name(0))
    print('Compute Cap.    :', torch.cuda.get_device_capability(0))

# lpips is required by Specular-Gaussians metrics but not always present in thesis_env
ensure_importable('lpips')

# torch_scatter ships a compiled extension pinned to a specific CUDA build; if the
# installed wheel doesn't match this env's torch+CUDA (e.g. "libcudart.so.12" missing),
# reinstall the wheel matching the *actual* installed torch version.
ensure_importable('torch_scatter', extra_pip_args=['-f', f'https://data.pyg.org/whl/torch-{torch.__version__}.html'])

# This codebase needs the anchor-based rasterizer fork (adds GaussianRasterizer.visible_filter,
# used by prefilter_voxel). A generic/vanilla diff_gaussian_rasterization build lacks that method.
try:
    import diff_gaussian_rasterization as _dgr
    if not hasattr(_dgr.GaussianRasterizer, 'visible_filter'):
        raise AttributeError("installed diff_gaussian_rasterization is missing 'visible_filter' (wrong fork/build)")
except Exception as e:
    print(f'diff_gaussian_rasterization: rebuilding ({e})')
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'diff_gaussian_rasterization'], capture_output=True)
    build_and_install_rasterizer()

import diff_gaussian_rasterization
import simple_knn
import lpips
import torch_scatter
assert hasattr(diff_gaussian_rasterization.GaussianRasterizer, 'visible_filter'), \
    'diff_gaussian_rasterization still missing visible_filter after rebuild'
print('rasterizer      : OK (visible_filter present)')
print('simple-knn      : OK')
print('lpips           : OK')
print('torch_scatter   : OK')

try:
    import huggingface_hub
    print('huggingface_hub : OK')
except ImportError:
    print('huggingface_hub : MISSING (will auto-install during the upload step)')

## c03 — Verify Dataset Path
Checks that the Synthetic Specular source dataset is available on the server.

In [ ]:
# ── Verify Dataset Directory ──────────────────────────────────────────────────
import os

src_root = "/home/ghp4hc/datasets/datasets/synthetic_specular"
assert os.path.exists(src_root), f"Dataset path not found at {src_root}! Check that the dataset is downloaded."
print(f"✅ Found dataset source root: {src_root}")

print(f"\n📂 Source datasets directory content:")
print(os.listdir(src_root))

## c04 — Verify Synthetic Specular Dataset Layout
Validates all 8 scenes and checks for `transforms_train.json` or `transforms.json`.

In [ ]:
# ── Verify Synthetic Specular Layout (all 8 scenes) ───────────────────────────
import os

src_root = "/home/ghp4hc/datasets/datasets/synthetic_specular"
SYNTHETIC_SCENES = [
    "ashtray", "dishes", "headphone", "jupyter",
    "lock", "plane", "record", "teapot"
]

print(f"Verifying layouts directly under {src_root} ...")
missing = []
for scene in SYNTHETIC_SCENES:
    scene_dir = os.path.join(src_root, scene)
    if not os.path.isdir(scene_dir):
        status = "MISSING (scene folder not found)"
        missing.append(scene)
    else:
        tf_train = os.path.join(scene_dir, "transforms_train.json")
        tf_single = os.path.join(scene_dir, "transforms.json")
        if os.path.isfile(tf_train):
            status = "OK (transforms_train.json found)"
        elif os.path.isfile(tf_single):
            status = "OK (transforms.json found)"
        else:
            status = f"MISSING (neither transforms_train.json nor transforms.json found; has: {sorted(os.listdir(scene_dir))[:6]})"
            missing.append(scene)

    print(f"  {scene:<12s} {status}")

print()
if missing:
    print(f"⚠️  {len(missing)}/{len(SYNTHETIC_SCENES)} scene(s) missing: {missing}")
    print("    run_synthetic_specular.sh will skip these scenes during the sweep.")
else:
    print(f"✅ All {len(SYNTHETIC_SCENES)} scenes are verified and available. Ready for training!")

## c05 — Run Specular-Gaussians Synthetic Specular Sweep (`run_synthetic_specular.sh`)
Launches the training sweep using `thesis_env` and system CUDA paths.

In [ ]:
# ── Run the sweep ──────────────────────────────────────────────────────────────
import subprocess
import os
import sys

LOGFILE = os.path.join(os.path.dirname(REPO_ROOT), "synthetic_specular_specular_gaussians_run.log")
DATA_ROOT = "/home/ghp4hc/datasets/datasets/synthetic_specular"

venv_bin = os.path.dirname(sys.executable)
cuda_home = find_cuda_home()
print(f"CUDA_HOME path: {cuda_home}")

cmd = f'''
export PATH={venv_bin}:{cuda_home}/bin:$PATH
export LD_LIBRARY_PATH={cuda_home}/lib64:$LD_LIBRARY_PATH
export CUDA_VISIBLE_DEVICES=0
export DATA_ROOT={DATA_ROOT}
cd "{REPO_ROOT}"
bash run_synthetic_specular.sh > "{LOGFILE}" 2>&1
'''
r = subprocess.run(['bash', '-c', cmd])

print(f"--- tail of {LOGFILE} ---")
if os.path.exists(LOGFILE):
    with open(LOGFILE, 'r') as f:
        lines = f.readlines()
        print(''.join(lines[-100:]))
if r.returncode != 0:
    print(f"⚠️  run_synthetic_specular.sh stopped early (STOP_ON_ERROR=True) -- check logs above or at {LOGFILE}")

## c06 — Quantitative Results Summary
Parses `results.json` and point cloud data from the output directory to print a formatted metrics table.

In [ ]:
# ── Quantitative Results Summary ──────────────────────────────────────────────
import json
import os

OUTPUT_ROOT = os.path.join(REPO_ROOT, "output", "synthetic_specular")

def fmt(x, nd=4):
    return f"{x:.{nd}f}" if isinstance(x, (int, float)) else "-"

def count_gaussians(scene_dir):
    ply_path = os.path.join(scene_dir, "point_cloud", "iteration_30000", "point_cloud.ply")
    if os.path.exists(ply_path):
        try:
            with open(ply_path, 'rb') as f:
                for line in f:
                    line_str = line.decode('ascii', errors='ignore')
                    if line_str.startswith("element vertex"):
                        return int(line_str.split()[2])
        except Exception:
            pass
    return "-"

header = f"{'scene':<12s}{'PSNR':>10s}{'SSIM':>10s}{'LPIPS':>10s}{'#Gaussians':>14s}"
print(header)
print("-" * len(header))

psnr_list, ssim_list, lpips_list = [], [], []

for scene in SYNTHETIC_SCENES:
    out_dir = os.path.join(OUTPUT_ROOT, scene)
    results_path = os.path.join(out_dir, "results.json")

    if not os.path.exists(results_path):
        print(f"{scene:<12s}  (no results.json -- skipped, or sweep did not reach this scene)")
        continue

    with open(results_path) as f:
        results = json.load(f)

    iter_key = next(iter(results.keys())) if results else None
    metrics = results.get(iter_key, {}) if iter_key else {}

    psnr_val = metrics.get("PSNR")
    ssim_val = metrics.get("SSIM")
    lpips_val = metrics.get("LPIPS")
    n_gauss = count_gaussians(out_dir)

    if isinstance(psnr_val, (int, float)):
        psnr_list.append(psnr_val)
    if isinstance(ssim_val, (int, float)):
        ssim_list.append(ssim_val)
    if isinstance(lpips_val, (int, float)):
        lpips_list.append(lpips_val)

    print(f"{scene:<12s}{fmt(psnr_val):>10s}{fmt(ssim_val):>10s}{fmt(lpips_val):>10s}{str(n_gauss):>14s}")

print("-" * len(header))
if psnr_list:
    avg_psnr = sum(psnr_list) / len(psnr_list)
    avg_ssim = sum(ssim_list) / len(ssim_list)
    avg_lpips = sum(lpips_list) / len(lpips_list)
    print(f"{'Average':<12s}{fmt(avg_psnr):>10s}{fmt(avg_ssim):>10s}{fmt(avg_lpips):>10s}{'-':>14s}")

## c07 — Packaging Submission ZIP
Zips the outputs directory into `specular_gaussians_output_synthetic_specular.zip`.

In [ ]:
# ── Packaging Submission ZIP ──────────────────────────────────────────────────
import shutil
import os

src = os.path.join(REPO_ROOT, "output", "synthetic_specular")
out = os.path.join(os.path.dirname(REPO_ROOT), "specular_gaussians_output_synthetic_specular")

if os.path.isdir(src):
    print(f"📦 Zipping folder '{src}' -> '{out}.zip' ...")
    if os.path.exists(out + ".zip"):
        os.remove(out + ".zip")
    shutil.make_archive(out, "zip", src)
    print("archived:", out + ".zip", round(os.path.getsize(out + ".zip") / 1e6, 1), "MB")
else:
    print(f"❌ ERROR: Output folder not found at '{src}'.")

## c08 — Hugging Face Results Upload & Auto-Cleanup
Uploads the zipped results archive to `DiBiay/specular_gaussians-synthetic_specular-result` and purges local zip & output files to save disk space.

In [ ]:
# ── Upload Results Zip to Hugging Face & Purge Local Files ─────────────────────
import os
import sys
import shutil
import subprocess

try:
    from huggingface_hub import HfApi
except ImportError:
    print("huggingface_hub not found. Installing via pip...")
    subprocess.run([sys.executable, "-m", "pip", "install", "huggingface_hub", "--proxy", PROXY], check=True)
    from huggingface_hub import HfApi

HF_TOKEN = os.environ.get('HF_TOKEN', '').strip()
HF_REPO = os.environ.get('HF_REPO', 'DiBiay/specular_gaussians-synthetic_specular-result')

zip_path = os.path.join(os.path.dirname(REPO_ROOT), 'specular_gaussians_output_synthetic_specular.zip')
if not os.path.isfile(zip_path):
    zip_path = None

if not HF_TOKEN:
    HF_TOKEN = input("Enter your Hugging Face Access Token (WRITE permission required): ").strip()

if zip_path and HF_TOKEN:
    print(f"🔍 Checking repository '{HF_REPO}' status...")
    api = HfApi()

    try:
        api.repo_info(repo_id=HF_REPO, repo_type="dataset", token=HF_TOKEN)
        print(f"✅ Repository '{HF_REPO}' already exists.")
    except Exception:
        print(f"➕ Creating private dataset repository '{HF_REPO}'...")
        try:
            api.create_repo(repo_id=HF_REPO, repo_type="dataset", token=HF_TOKEN, private=True)
            print(f"🎉 Created private repository '{HF_REPO}' successfully.")
        except Exception as e:
            print(f"⚠️  Could not create repository: {e}")

    print(f"📤 Uploading '{zip_path}' to dataset repository '{HF_REPO}'...")
    try:
        api.upload_file(
            path_or_fileobj=zip_path,
            path_in_repo="synthetic_specular.zip",
            repo_id=HF_REPO,
            repo_type="dataset",
            token=HF_TOKEN,
        )
        print("🎉 [SUCCESS] Results archive uploaded to Hugging Face successfully!")

        out_dir = os.path.join(REPO_ROOT, "output", "synthetic_specular")
        print(f"🗑️ Cleaning up local zip file '{zip_path}' and output directory '{out_dir}'...")
        if os.path.exists(zip_path):
            os.remove(zip_path)
        if os.path.isdir(out_dir):
            shutil.rmtree(out_dir, ignore_errors=True)
        print("✅ Local disk space freed!")
    except Exception as e:
        print(f"❌ [ERROR] Hugging Face upload failed: {e}")
else:
    print(f"⚠️  Skipping upload: zip file not found or HF_TOKEN not set.")
    if not zip_path:
        print(f"    Expected zip at: {os.path.join(os.path.dirname(REPO_ROOT), 'specular_gaussians_output_synthetic_specular.zip')}")